In [28]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import random

from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

# plot correlation
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr, spearmanr

# Set publication-quality style
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 12,
    'axes.linewidth': 1.5,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'axes.titlesize': 16,
    'pdf.fonttype': 42
})

# Set random seed for reproducibility
seed = 0
random.seed(seed)
np.random.seed(seed)    
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

GX = pd.read_csv('~/lab_projects/splicing/for_kai/V4_data_with_offset_major_event_only/241126_gene_expression_all.csv')
# GX = pd.read_csv('~/lab_projects/splicing/for_kai/V4_data_with_offset_major_event_only/241126_gene_expression_RBP_only.csv')
PSI = pd.read_csv('~/lab_projects/splicing/for_kai/V4_data_with_offset_major_event_only/250407_measurement_ratio.csv')
Barcode = pd.read_csv('~/lab_projects/splicing/for_kai/V4_data_with_offset_major_event_only/250407_twist_barcodes_with_splice_site_annotation.csv')

In [29]:
GX_dict = GX.groupby("condition").apply(
    lambda x: np.array(x.drop(columns=["Unnamed: 0", "condition"]).values.flatten().tolist())
).to_dict()

/tmp/ipykernel_15265/3826946481.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  GX_dict = GX.groupby("condition").apply(


In [30]:
# make index as the barcode
Barcode = Barcode.set_index("index_offset")
Barcode = Barcode.drop(columns=["Unnamed: 0"])
Barcode


,padded_sequence,exon_intron_map,is_splice_acceptor,is_splice_donor
index_offset,,,,
ENSG00000000003.15;TSPAN6;chrX-100632484-100632568-100630758-100630866-100633404-100633539__0:0:0,CTTCGACACCGAGCTCGATATGATCGAAGTATTTATTACCATAAAG...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000000003.15;TSPAN6;chrX-100633930-100634029-100632484-100632568-100635177-100635252__0:0:0,GCTTCGACACCGAGCTCGTCGAGAACTTATTTGACCTGAAACCAAA...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000000003.15;TSPAN6;chrX-100635177-100635252-100633930-100634029-100635557-100635746__0:0:0,GCTTCGACACCGAGCTCGAGACGACCATTATTTTTTCTTTGACTCC...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000000419.14;DPM1;chr20-50945736-50945762-50942030-50942126-50945846-50945861__0:0:0,TGAGATTGAATCCAGGAAATGAAGCTTCGACACCGAGCTCGTTAGC...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000000419.14;DPM1;chr20-50948628-50948662-50945736-50945923-50955185-50955285__0:0:0,CTTCGACACCGAGCTCGGTGCAACTATATTTCTATTAAAGTGAGTA...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
...,...,...,...,...
ENSG00000288710.1;RP11-386G11.12;chr12-49005461-49005543-49005305-49005364-49005742-49005852__0:0:0,CTTCGACACCGAGCTCGACCCAACATGCCCAAACACTGTTCTTTTT...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000288710.1;RP11-386G11.12;chr12-49022279-49022353-49022042-49022151-49022589-49022875__0:0:0,CTTCGACACCGAGCTCGACTGCCTGAGTCTCCTACCTGATCCCACA...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000288717.1;RP11-852E15.4;chr3-46000912-46000955-45995957-45996035-46001083-46001283__0:0:0,GCTTCGACACCGAGCTCGAGATGAAGGCAAGGTTAGGGGTATCCGT...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...


In [31]:
# make index as the barcode
PSI = PSI.set_index("index_offset")
PSI = PSI.drop(columns=["Unnamed: 0"])
PSI

,769P,786O,8MGBA,A172,A375,ACHN,CAL120,COGN278,COLO783,DAOY,...,SF126,SKNAS,SNU398,SNU423,SNU449,SNUC4,T47D,TOV21G,U251MG,VMRCRCZ
index_offset,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003.15;TSPAN6;chrX-100632484-100632568-100630758-100630866-100633404-100633539__0:0:0,5.855052,6.955650,4.786596,5.066832,4.343257,5.347252,6.247928,5.172890,9.967226,8.377934,...,9.967226,4.703436,5.544321,3.755662,9.967226,9.967226,6.139551,9.967226,9.967226,9.967226
ENSG00000000003.15;TSPAN6;chrX-100633930-100634029-100632484-100632568-100635177-100635252__0:0:0,4.160823,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,...,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226
ENSG00000000003.15;TSPAN6;chrX-100635177-100635252-100633930-100634029-100635557-100635746__0:0:0,NaN,1.069751,-0.060542,-0.147135,-0.503730,1.844684,NaN,1.994607,9.967226,2.121991,...,-0.095157,0.997839,1.779734,NaN,-1.286800,1.089583,-0.147135,0.678072,0.617465,0.828326
ENSG00000000419.14;DPM1;chr20-50945736-50945762-50942030-50942126-50945846-50945861__0:0:0,1.252026,1.712718,3.996940,2.492914,-0.233995,2.835563,1.672780,4.786596,0.158698,1.149682,...,2.632603,4.829909,1.633412,4.160823,2.093702,2.248800,3.801190,1.163165,0.794913,1.641242
ENSG00000000419.14;DPM1;chr20-50948628-50948662-50945736-50945923-50955185-50955285__0:0:0,NaN,NaN,0.280483,NaN,-3.321928,-1.581126,NaN,NaN,NaN,NaN,...,NaN,-1.116179,NaN,NaN,NaN,NaN,-1.736966,-2.248800,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000288710.1;RP11-386G11.12;chr12-49005461-49005543-49005305-49005364-49005742-49005852__0:0:0,-0.684164,-0.135578,-0.474069,-2.756957,-2.731449,0.118249,-0.714717,0.503730,0.991361,0.292127,...,-0.895303,-0.819826,0.014413,-1.203909,-2.862496,-0.794913,-2.822237,-2.822237,-0.971986,-0.129800
ENSG00000288710.1;RP11-386G11.12;chr12-49022279-49022353-49022042-49022151-49022589-49022875__0:0:0,-0.539463,0.895303,0.095157,-0.391524,-1.238212,1.116179,-0.828326,1.192349,-0.072078,0.907996,...,1.328998,-0.060542,1.581126,-0.379787,-1.422312,0.462233,-0.901645,-0.491853,0.690262,0.456320
ENSG00000288717.1;RP11-852E15.4;chr3-46000912-46000955-45995957-45996035-46001083-46001283__0:0:0,-0.309611,0.545434,0.426815,1.036911,-1.541097,0.521577,-0.228193,1.176697,0.170265,2.681408,...,0.665905,-0.158698,0.303781,-0.112475,-0.474069,0.770115,0.257222,0.794913,0.344648,2.142811


In [32]:
num_na = PSI.isna().sum().sum()
num_non_na = PSI.size - num_na

print(f"NA: {num_na}")
print(f"~NA: {num_non_na}")

NA: 119406
~NA: 1982178


In [33]:
def onehot_encode_sequences(sequences):
    """
    One-hot encodes a list of DNA sequences.

    Args:
        sequences (list of str): A list of DNA sequences, each containing characters 'A', 'G', 'C', 'T'.

    Returns:
        np.ndarray: A 3D numpy array where each sequence is one-hot encoded along the last axis.
    """
    # Define a mapping for bases
    base_mapping = {'A': 0, 'G': 1, 'C': 2, 'T': 3, 'N': 4}
    n_sequences = len(sequences)
    sequence_length = len(sequences[0]) if sequences else 0
    
    # Initialize the one-hot encoded array
    onehot_array = np.zeros((n_sequences, sequence_length, 4), dtype=int)
    
    for i, seq in enumerate(sequences):
        for j, base in enumerate(seq):
            if base in base_mapping:  # Ensure valid base
                onehot_array[i, j, base_mapping[base]] = 1
                
    return onehot_array

Barcode_onehot = onehot_encode_sequences(list(Barcode['padded_sequence'].values))
# Barcode_onehot = np.concatenate((Barcode_onehot, twist_array), axis=-1)
print(Barcode_onehot.shape)
Barcode_dict = dict(zip(Barcode.index, Barcode_onehot))


(43783, 250, 4)


In [34]:
class PSIDataset(Dataset):
    def __init__(self, PSI_df, Barcode_dict, GX_dict):
        self.samples = []
    
        for barcode in PSI_df.index:
            if barcode not in Barcode_dict:
                continue  
            
            barcode_array = Barcode_dict[barcode] 
            
            for celltype in PSI_df.columns:
                if celltype not in GX_dict:
                    continue 
                
                gx_array = GX_dict[celltype] 
                psi_value = PSI_df.loc[barcode, celltype]
                
                if not np.isnan(psi_value):
                    self.samples.append((celltype, barcode, barcode_array, gx_array, psi_value))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        celltype, barcode, barcode_array, gx_array, psi_value = self.samples[idx]
        return celltype, barcode, barcode_array, gx_array, psi_value
    
def split_dataset_by_barcode(dataset, test_size=0.2, random_state=42):
    """
    Split the dataset into train and test subsets based on barcode.
    Params:
    - dataset: PSIDataset
    - test_size: ratio
    - random_state
    
    Return:
    - train_dataset
    - test_dataset
    """
    # Extract all barcodes
    barcodes = np.array([sample[1] for sample in dataset.samples])
    
    # Get unique barcodes and split
    unique_barcodes = np.unique(barcodes)
    
    train_barcodes, test_barcodes = train_test_split(
        unique_barcodes, test_size=test_size, random_state=random_state
    )
    
    print(f"Train barcodes: {train_barcodes[:5]}")
    print(f"Test barcodes: {test_barcodes[:5]}") 
    
    train_barcodes_set = set(train_barcodes)
    test_barcodes_set = set(test_barcodes)
    
    # Get indices for train and test
    train_indices = np.where(np.isin(barcodes, list(train_barcodes_set)))[0]
    test_indices = np.where(np.isin(barcodes, list(test_barcodes_set)))[0]
    print(f"Train indices: {len(train_indices)}, Test indices: {len(test_indices)}")
    
    # Construct Subsets
    train_dataset = Subset(dataset, train_indices)
    test_dataset = Subset(dataset, test_indices)
    
    return train_dataset, test_dataset

def split_dataset_by_celltype(dataset, test_size=0.2, random_state=42):
    """
    Split the dataset into train and test subsets based on celltype.
    Params:
    - dataset: PSIDataset
    - test_size: ratio
    - random_state
    
    Return:
    - train_dataset
    - test_dataset
    """
    celltypes = np.array([sample[0] for sample in dataset.samples])
    
    unique_celltypes = np.unique(celltypes)
    train_celltypes, test_celltypes = train_test_split(
        unique_celltypes, test_size=test_size, random_state=random_state
    )
    
    print(f"Train celltypes: {len(train_celltypes)}, Test celltypes: {len(test_celltypes)}")
    
    train_indices = [i for i, celltype in enumerate(celltypes) if celltype in train_celltypes]
    test_indices = [i for i, celltype in enumerate(celltypes) if celltype in test_celltypes]
    
    train_dataset = Subset(dataset, train_indices)
    test_dataset = Subset(dataset, test_indices)
    
    assert len(train_dataset) == len(train_indices), "Train dataset size mismatch!"
    assert len(test_dataset) == len(test_indices), "Test dataset size mismatch!"
    
    return train_dataset, test_dataset

def split_dataset_random(dataset, test_size=0.2, random_state=42):

    dataset_size = len(dataset)
    
    indices = list(range(dataset_size))
    
    train_indices, test_indices = train_test_split(
        indices, test_size=test_size, random_state=random_state
    )
    
    train_dataset = Subset(dataset, train_indices)
    test_dataset = Subset(dataset, test_indices)
    
    return train_dataset, test_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

lr = 1e-4
epochs = 5
batch_size = 1024

dataset = PSIDataset(PSI, Barcode_dict, GX_dict)
dataset_size = len(dataset)
print(f"Total samples: {dataset_size}")

# # train_dataset, test_dataset = split_dataset_random(dataset, test_size=0.2, random_state=42)
# train_dataset, test_dataset = split_dataset_by_barcode(dataset, test_size=0.2, random_state=42)

# # train_dataset, test_dataset = split_dataset_by_celltype(dataset, test_size=0.2, random_state=42)
# print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8)
# test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=8)

Total samples: 1828278


## Convolutional Neural Network

In [35]:
class GexFullyConnected(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(in_features=19221, out_features=1024)
        self.fc2 = nn.Linear(in_features=1024, out_features=256)
        self.fc3 = nn.Linear(in_features=256, out_features=128)
        self.BN1 = nn.BatchNorm1d(1024)
        self.BN2 = nn.BatchNorm1d(256) 
        self.dropout = nn.Dropout(0.2)     

    def forward(self, x):
        x = F.relu(self.BN1(self.fc1(x)))
        x = F.relu(self.BN2(self.fc2(x)))
        x = F.relu(self.fc3(x))
        return x

class ResNetCNN(nn.Module):
    def __init__(self, channels, dropout=0.2):
        super().__init__() 
        
        self.conv1_5 =  nn.Conv1d(channels, channels, kernel_size=5, padding=2)  
        self.conv1_11 = nn.Conv1d(channels, channels, kernel_size=11, padding=5)
        self.conv1_21 = nn.Conv1d(channels, channels, kernel_size=21, padding=10)

        self.batchnorm1 = nn.BatchNorm1d(channels * 3) 
        self.dropout = nn.Dropout(dropout)
        self.conv_merge = nn.Conv1d(channels * 3, channels, kernel_size=1)

        self.conv2 = nn.Conv1d(channels, channels, kernel_size=5, padding=2)
        self.batchnorm2 = nn.BatchNorm1d(channels)

    def forward(self, x):
        residual = x 
        x1 = self.conv1_5(x)
        x2 = self.conv1_11(x)
        x3 = self.conv1_21(x)

        x = torch.cat([x1, x2, x3], dim=1) 

        x = self.dropout(F.relu(self.batchnorm1(x)))
        x = self.conv_merge(x) 
        x = self.dropout(F.relu(self.batchnorm2(self.conv2(x))))
        
        x += residual  
        return x

class SequenceCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.dropout = nn.Dropout(0.2)
        
        self.conv1 = nn.Conv1d(in_channels=4, out_channels=128, kernel_size=1, padding='same')
        self.batchnorm1 = nn.BatchNorm1d(128)
    
        self.resnetCNN1 = nn.Sequential(
            ResNetCNN(128),
            ResNetCNN(128),
            ResNetCNN(128),
        )
        
        self.maxpool = nn.MaxPool1d(kernel_size=2)
        
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=64, kernel_size=1, padding='same')
        self.batchnorm2 = nn.BatchNorm1d(64)
        
        self.resnetCNN2 = nn.Sequential(
            ResNetCNN(64),
            ResNetCNN(64),
            ResNetCNN(64),
        )
        
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=16, kernel_size=1, padding='same')
        self.batchnorm3 = nn.BatchNorm1d(16)
        
        self.resnetCNN3 = nn.Sequential(
            ResNetCNN(16),
            ResNetCNN(16),
            ResNetCNN(16),
        )

        self.conv4 = nn.Conv1d(in_channels=128, out_channels=16, kernel_size=1, padding='same')
        self.maxpool8 = nn.MaxPool1d(kernel_size=8)
        
        # Linear layers
        self.fc1 = nn.Linear(in_features=496, out_features=128)
        
    def forward(self, x):
        x = self.dropout(F.relu(self.batchnorm1(self.conv1(x))))
        residual = x
        x_residual = x
        x = self.resnetCNN1(x)
        x += residual
        x = self.maxpool(x)
        
        x = self.dropout(F.relu(self.batchnorm2(self.conv2(x))))
        residual = x
        x = self.resnetCNN2(x)
        x += residual
        x = self.maxpool(x)
        
        x = self.dropout(F.relu(self.batchnorm3(self.conv3(x))))
        residual = x
        x = self.resnetCNN3(x)
        x += residual
        x = self.maxpool(x)
        
        x_residual = self.conv4(x_residual)
        x_residual = self.maxpool8(x_residual)
        x = x + x_residual
        
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        
        return x

class Predictor_gene_barcode(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.Barcode_model = SequenceCNN()
        self.GX_model = GexFullyConnected()

        self.fc1 = nn.Linear(in_features=128, out_features=64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(in_features=64, out_features=1)
        
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, barcode, gx, condition):
        
        z_barcode = self.Barcode_model(barcode)
        
        z_gx = self.GX_model(gx)
        x = z_barcode + z_gx + condition
        x = self.dropout(F.leaky_relu(self.bn1(self.fc1(x))))
        x = self.fc2(x)
        
        return x
    
class Predictor_barcode(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.Barcode_model = SequenceCNN()

        self.fc1 = nn.Linear(in_features=128, out_features=64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(in_features=64, out_features=1)
        
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, barcode):
        
        x = self.Barcode_model(barcode)
        x = self.dropout(F.leaky_relu(self.bn1(self.fc1(x))))
        x = self.fc2(x)
        
        return x

## Evaluation for model_gene_barcode

In [36]:
import os
output_dir = "/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/"
seed_set = range(10)
all_correlations = []

for seed in seed_set:
    random.seed(seed)
    np.random.seed(seed)    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Create train/test split using current seed
    train_dataset, test_dataset = split_dataset_by_barcode(dataset, test_size=0.2, random_state=seed)
    print(f"Seed {seed} - Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=8)

    model_barcode = Predictor_barcode()
    model_barcode.load_state_dict(torch.load(f"/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/model_barcode_PSI_no_mut_5epoch_seed{seed}.pth"))
    model_barcode.eval()
    model_barcode = model_barcode.to(device)

    MSELoss = nn.MSELoss()
    optimizer = torch.optim.Adam(model_barcode.parameters(), lr=lr)

    model_gene_barcode = Predictor_gene_barcode()
    model_gene_barcode.load_state_dict(torch.load(f"/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/model_barcode_gene_PSI_no_mut_5epoch_seed{seed}.pth"))
    model_gene_barcode.eval()
    model_gene_barcode = model_gene_barcode.to(device)
    
    predict_psi = []
    real_psi = []
    psi_residuals = []
    celltypes = []
    barcode_names = []
    
    with torch.no_grad():
        psi_loss = 0
        recon_loss = 0
        for i, (celltype, barcode_name, barcode, gx, psi) in enumerate(test_loader):
            barcode = barcode.to(device).float()
            barcode = barcode.permute(0, 2, 1)
            gx = gx.to(device).float()
            psi = psi.to(device).float()
        
            condition = model_barcode.Barcode_model(barcode)
            psi_pretrain = model_barcode(barcode)
        
            psi_residual = model_gene_barcode(barcode, gx, condition)
            psi_loss += MSELoss(psi_residual.squeeze(), psi-psi_pretrain.squeeze()).item()
            
            predict_psi.extend(psi_residual.squeeze().cpu().numpy()+psi_pretrain.squeeze().cpu().numpy())
            psi_residuals.extend(psi_residual.squeeze().cpu().numpy())
            real_psi.extend(psi.cpu().numpy())
            celltypes.extend(celltype)
            barcode_names.extend(barcode_name)

    pearson_correlation, _ = pearsonr(real_psi, predict_psi)
    spearman_correlation, _ = spearmanr(real_psi, predict_psi)
    all_correlations.append((pearson_correlation, spearman_correlation))
    
    print(f"\nSeed {seed}:")
    print(f"Pearson Correlation: {pearson_correlation:.4f}")
    print(f"Spearman Correlation: {spearman_correlation:.4f}")

    # Save predictions for this seed
    psi_results_df = pd.DataFrame({
        'Real_PSI': real_psi,
        'Predicted_PSI': predict_psi,
        'PSI_Residuals': psi_residuals,
        'Celltype': celltypes,
        'Barcode': barcode_names
    })
    psi_results_df.to_csv(os.path.join(output_dir, f'psi_predictions_barcode_gene_seed{seed}.csv'), index=False)
    print(f"Saved PSI predictions to {os.path.join(output_dir, f'psi_predictions_barcode_gene_seed{seed}.csv')}")

    # Create and save scatter plot for this seed
    plt.figure(figsize=(8, 8), dpi=300)
    plt.scatter(real_psi, predict_psi, alpha=0.5, s=5, c='#336695', edgecolors='none', marker='o', rasterized=True)
    plt.plot([-10, 10], [-10, 10], 'k--', alpha=0.5, zorder=0)
    plt.xlabel("Observed PSI", fontsize=14, fontweight='bold')
    plt.ylabel("Predicted PSI", fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.xlim(-10, 10)
    plt.ylim(min(predict_psi), max(predict_psi))
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['left'].set_linewidth(1.5)
    plt.gca().spines['bottom'].set_linewidth(1.5)
    plt.title(f"Prediction Performance on Wild-type Data (Seed {seed})", fontsize=16, fontweight='bold', pad=20)
    stats_text = f"Pearson r = {pearson_correlation:.3f}\nSpearman ρ = {spearman_correlation:.3f}"
    plt.text(0.05, 0.95, stats_text,
             transform=plt.gca().transAxes,
             bbox=dict(facecolor='white', edgecolor='none', alpha=0.7),
             fontsize=12,
             verticalalignment='top')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'predicted_vs_real_psi_gene_barcode_seed{seed}.pdf'), 
                bbox_inches='tight', 
                dpi=300)
    plt.close()

# After all seeds are processed, create summary statistics and plots
pearson_values = [c[0] for c in all_correlations]
spearman_values = [c[1] for c in all_correlations]

# Save correlation values to CSV
correlation_df = pd.DataFrame({
    'Pearson': pearson_values,
    'Spearman': spearman_values
})
correlation_df.to_csv(os.path.join(output_dir, 'correlation_values_model_barcode_gene.csv'), index=False)

# Calculate means and stds
pearson_mean = np.mean(pearson_values)
pearson_std = np.std(pearson_values)
spearman_mean = np.mean(spearman_values)
spearman_std = np.std(spearman_values)

# Create and save box plot
plt.figure(figsize=(8, 6))
box_data = [pearson_values, spearman_values]
plt.boxplot(box_data, labels=['Pearson', 'Spearman'])
plt.ylabel('Correlation')
plt.title('Correlation Scores Across Seeds')
x_positions = [1, 2]
for i, data in enumerate([pearson_values, spearman_values]):
    plt.scatter([x_positions[i]] * len(data), data, alpha=0.5, color='black')
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['top'].set_visible(False)
print(f"\nAverage across all seeds:")
print(f"Pearson Correlation: {pearson_mean:.4f} ± {pearson_std:.4f}")
print(f"Spearman Correlation: {spearman_mean:.4f} ± {spearman_std:.4f}")
plt.savefig(os.path.join(output_dir, 'correlation_boxplot.pdf'), 
            bbox_inches='tight', 
            dpi=300)
plt.close()


Train barcodes: ['ENSG00000184156.17;KCNQ3;chr8-132134325-132134388-132132179-132132264-132137884-132138016__0:0:0'
 'ENSG00000105613.10;MAST1;chr19-12858361-12858441-12852327-12852395-12858530-12858739__0:0:0'
 'ENSG00000171109.19;MFN1;chr3-179365117-179365225-179364296-179364405-179367438-179367498__0:0:0'
 'ENSG00000203321.2;CARNMT1-AS1;chr9-74991620-74991674-74991164-74991285-74995997-74996293__0:0:0'
 'ENSG00000104823.9;ECH1;chr19-38817315-38817364-38817064-38817129-38831077-38831166__0:0:0']
Test barcodes: ['ENSG00000133106.15;EPSTI1;chr13-42963254-42963338-42953947-42954021-42969093-42969177__0:0:0'
 'ENSG00000183671.12;GPR1;chr2-206203893-206203948-206176565-206177275-206213306-206213371__0:0:0'
 'ENSG00000145741.16;BTF3;chr5-73504346-73504403-73502487-73503117-73505191-73505435__0:0:0'
 'ENSG00000124596.17;OARD1;chr6-41075266-41075349-41071595-41071675-41097712-41097787__0:0:0'
 'ENSG00000096063.16;SRPK1;chr6-35920467-35920528-35890894-35891013-35921043-35921072__0:0:0']
Train

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Train barcodes: ['ENSG00000138152.10;BTBD16;chr10-122297767-122297837-122291079-122291194-122307188-122307308__0:0:0'
 'ENSG00000123411.15;IKZF4;chr12-56018125-56018177-56007668-56007734-56025053-56025158__0:0:0'
 'ENSG00000146263.12;MMS22L;chr6-97246627-97246690-97233860-97233980-97267871-97268002__0:0:0'
 'ENSG00000122126.17;OCRL;chrX-129567253-129567363-129565771-129565883-129569263-129569399__0:0:0'
 'ENSG00000137845.15;ADAM10;chr15-58643885-58643978-58640776-58640960-58646054-58646204__0:0:0']
Test barcodes: ['ENSG00000066923.18;STAG3;chr7-100182722-100182839-100182089-100182192-100189444-100189596__0:0:0'
 'ENSG00000100003.18;SEC14L2;chr22-30406341-30406385-30399642-30399718-30407094-30407154__0:0:0'
 'ENSG00000100344.11;PNPLA3;chr22-43928823-43928889-43926934-43927167-43932877-43933087__0:0:0'
 'ENSG00000183431.12;SF3A3;chr1-37978987-37979055-37976883-37976953-37979464-37979533__0:0:0'
 'ENSG00000021826.17;CPS1;chr2-210668185-210668284-210663122-210663197-210674901-210674961__0:

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Train barcodes: ['ENSG00000223458.2;LMO7DN-IT1;chr13-75878714-75878794-75876885-75876965-75879703-75879782__0:0:0'
 'ENSG00000164692.18;COL1A2;chr7-94409722-94409821-94409563-94409608-94410241-94410295__0:0:0'
 'ENSG00000100033.17;PRODH;chr22-18921341-18921423-18919705-18919798-18925050-18925200__0:0:0'
 'ENSG00000172995.17;ARPP21;chr3-35708968-35709070-35706973-35707082-35715438-35715476__0:0:0'
 'ENSG00000145700.10;ANKRD31;chr5-75112512-75112600-75107520-75107617-75116565-75116681__0:0:0']
Test barcodes: ['ENSG00000164134.13;NAA15;chr4-139354025-139354098-139351504-139351611-139359742-139359857__0:0:0'
 'ENSG00000136878.14;USP20;chr9-129856306-129856360-129852539-129852636-129858049-129858112__0:0:0'
 'ENSG00000177830.18;CHID1;chr11-899339-899401-893426-893519-900003-900110__0:0:0'
 'ENSG00000262879.6;RP11-156P1.3;chr17-47007839-47007884-46998699-46998817-47021581-47021721__0:0:0'
 'ENSG00000149403.13;GRIK4;chr11-120898531-120898639-120875138-120875243-120905289-120905493__0:0:0']
Tr

...

Train barcodes: ['ENSG00000134690.11;CDCA8;chr1-37706977-37707064-37703251-37703347-37708321-37709713__0:0:0'
 'ENSG00000100578.18;KIAA0586;chr14-58512521-58512627-58508554-58508709-58540070-58540136__0:0:0'
 'ENSG00000102001.13;CACNA1F;chrX-49214158-49214269-49213818-49213902-49215085-49215244__0:0:0'
 'ENSG00000167306.20;MYO5B;chr18-49990438-49990520-49980443-49980553-49992287-49992431__0:0:0'
 'ENSG00000198677.12;TTC37;chr5-95506933-95506992-95503813-95503938-95513570-95513645__-19:0:0']
Test barcodes: ['ENSG00000141030.13;COPS3;chr17-17261599-17261692-17260373-17260474-17261965-17262106__0:0:0'
 'ENSG00000065413.20;ANKRD44;chr2-197187022-197187106-197147026-197147105-197310577-197310646__0:0:0'
 'ENSG00000001617.12;SEMA3F;chr3-50174051-50174114-50173792-50173953-50174230-50174350__0:0:0'
 'ENSG00000148187.18;MRRF;chr9-122291748-122291840-122270807-122271075-122322539-122323092__0:0:0'
 'ENSG00000131966.14;ACTR10;chr14-58207935-58208018-58202854-58202927-58208998-58209107__0:0:0']
T

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Train barcodes: ['ENSG00000068971.14;PPP2R5B;chr11-64928065-64928158-64927733-64927903-64928294-64928425__0:0:0'
 'ENSG00000123240.17;OPTN;chr10-13116266-13116340-13112452-13112635-13118887-13118980__0:-57:0'
 'ENSG00000272281.6;TPTE2P2;chr13-52229325-52229431-52227004-52227096-52253744-52253833__0:0:0'
 'ENSG00000047849.22;MAP4;chr3-47867245-47867338-47857430-47857512-47869213-47869327__0:0:0'
 'ENSG00000144867.13;SRPRB;chr3-133807745-133807823-133806552-133806703-133811116-133811199__0:0:0']
Test barcodes: ['ENSG00000105879.12;CBLL1;chr7-107753894-107753978-107753410-107753511-107758142-107758571__0:0:0'
 'ENSG00000140943.17;MBTPS1;chr16-84091731-84091848-84090874-84090942-84093710-84093821__0:0:0'
 'ENSG00000129460.16;NGDN;chr14-23470905-23470977-23469743-23470101-23475170-23475308__0:0:0'
 'ENSG00000205426.10;KRT81;chr12-52288360-52288456-52287983-52288148-52289214-52289275__0:0:0'
 'ENSG00000136811.16;ODF2;chr9-128492425-128492536-128487889-128488025-128492700-128492805__0:0:0']
T

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


Average across all seeds:
Pearson Correlation: 0.7302 ± 0.0063
Spearman Correlation: 0.7359 ± 0.0068


# Evaluation for model with high variance data for finetune

In [37]:
import os
output_dir = "/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/"
seed_set = range(10)
all_correlations = []

for seed in seed_set:
    random.seed(seed)
    np.random.seed(seed)    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Create train/test split using current seed
    train_dataset, test_dataset = split_dataset_by_barcode(dataset, test_size=0.2, random_state=seed)
    print(f"Seed {seed} - Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=8)

    model_barcode = Predictor_barcode()
    model_barcode.load_state_dict(torch.load(f"/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/model_barcode_PSI_no_mut_5epoch_seed{seed}.pth"))
    model_barcode.eval()
    model_barcode = model_barcode.to(device)

    MSELoss = nn.MSELoss()
    optimizer = torch.optim.Adam(model_barcode.parameters(), lr=lr)

    model_gene_barcode = Predictor_gene_barcode()
    model_gene_barcode.load_state_dict(torch.load(f"/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/model_barcode_gene_PSI_no_mut_5epoch_high_var_only_seed{seed}.pth"))
    model_gene_barcode.eval()
    model_gene_barcode = model_gene_barcode.to(device)
    
    predict_psi = []
    real_psi = []
    psi_residuals = []
    celltypes = []
    barcode_names = []
    
    with torch.no_grad():
        psi_loss = 0
        recon_loss = 0
        for i, (celltype, barcode_name, barcode, gx, psi) in enumerate(test_loader):
            barcode = barcode.to(device).float()
            barcode = barcode.permute(0, 2, 1)
            gx = gx.to(device).float()
            psi = psi.to(device).float()
        
            condition = model_barcode.Barcode_model(barcode)
            psi_pretrain = model_barcode(barcode)
        
            psi_residual = model_gene_barcode(barcode, gx, condition)
            psi_loss += MSELoss(psi_residual.squeeze(), psi-psi_pretrain.squeeze()).item()
            
            predict_psi.extend(psi_residual.squeeze().cpu().numpy()+psi_pretrain.squeeze().cpu().numpy())
            psi_residuals.extend(psi_residual.squeeze().cpu().numpy())
            real_psi.extend(psi.cpu().numpy())
            celltypes.extend(celltype)
            barcode_names.extend(barcode_name)

    pearson_correlation, _ = pearsonr(real_psi, predict_psi)
    spearman_correlation, _ = spearmanr(real_psi, predict_psi)
    all_correlations.append((pearson_correlation, spearman_correlation))
    
    print(f"\nSeed {seed}:")
    print(f"Pearson Correlation: {pearson_correlation:.4f}")
    print(f"Spearman Correlation: {spearman_correlation:.4f}")

    # Save predictions for this seed
    psi_results_df = pd.DataFrame({
        'Real_PSI': real_psi,
        'Predicted_PSI': predict_psi,
        'PSI_Residuals': psi_residuals,
        'Celltype': celltypes,
        'Barcode': barcode_names
    })
    psi_results_df.to_csv(os.path.join(output_dir, f'psi_predictions_barcode_gene_high_var_only_seed{seed}.csv'), index=False)
    print(f"Saved PSI predictions to {os.path.join(output_dir, f'psi_predictions_barcode_gene_high_var_only_seed{seed}.csv')}")

    # Create and save scatter plot for this seed
    plt.figure(figsize=(8, 8), dpi=300)
    plt.scatter(real_psi, predict_psi, alpha=0.5, s=5, c='#336695', edgecolors='none', marker='o', rasterized=True)
    plt.plot([-10, 10], [-10, 10], 'k--', alpha=0.5, zorder=0)
    plt.xlabel("Observed PSI", fontsize=14, fontweight='bold')
    plt.ylabel("Predicted PSI", fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.xlim(-10, 10)
    plt.ylim(min(predict_psi), max(predict_psi))
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['left'].set_linewidth(1.5)
    plt.gca().spines['bottom'].set_linewidth(1.5)
    plt.title(f"Prediction Performance on Wild-type Data (Seed {seed})", fontsize=16, fontweight='bold', pad=20)
    stats_text = f"Pearson r = {pearson_correlation:.3f}\nSpearman ρ = {spearman_correlation:.3f}"
    plt.text(0.05, 0.95, stats_text,
             transform=plt.gca().transAxes,
             bbox=dict(facecolor='white', edgecolor='none', alpha=0.7),
             fontsize=12,
             verticalalignment='top')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'predicted_vs_real_psi_gene_barcode_high_var_only_seed{seed}.pdf'), 
                bbox_inches='tight', 
                dpi=300)
    plt.close()

# After all seeds are processed, create summary statistics and plots
pearson_values = [c[0] for c in all_correlations]
spearman_values = [c[1] for c in all_correlations]

# Save correlation values to CSV
correlation_df = pd.DataFrame({
    'Pearson': pearson_values,
    'Spearman': spearman_values
})
correlation_df.to_csv(os.path.join(output_dir, 'correlation_values_model_barcode_gene_high_var_only.csv'), index=False)

# Calculate means and stds
pearson_mean = np.mean(pearson_values)
pearson_std = np.std(pearson_values)
spearman_mean = np.mean(spearman_values)
spearman_std = np.std(spearman_values)

# Create and save box plot
plt.figure(figsize=(8, 6))
box_data = [pearson_values, spearman_values]
plt.boxplot(box_data, labels=['Pearson', 'Spearman'])
plt.ylabel('Correlation')
plt.title('Correlation Scores Across Seeds')
x_positions = [1, 2]
for i, data in enumerate([pearson_values, spearman_values]):
    plt.scatter([x_positions[i]] * len(data), data, alpha=0.5, color='black')
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['top'].set_visible(False)
print(f"\nAverage across all seeds:")
print(f"Pearson Correlation: {pearson_mean:.4f} ± {pearson_std:.4f}")
print(f"Spearman Correlation: {spearman_mean:.4f} ± {spearman_std:.4f}")
plt.savefig(os.path.join(output_dir, 'correlation_boxplot_high_var_only.pdf'), 
            bbox_inches='tight', 
            dpi=300)
plt.close()


Train barcodes: ['ENSG00000184156.17;KCNQ3;chr8-132134325-132134388-132132179-132132264-132137884-132138016__0:0:0'
 'ENSG00000105613.10;MAST1;chr19-12858361-12858441-12852327-12852395-12858530-12858739__0:0:0'
 'ENSG00000171109.19;MFN1;chr3-179365117-179365225-179364296-179364405-179367438-179367498__0:0:0'
 'ENSG00000203321.2;CARNMT1-AS1;chr9-74991620-74991674-74991164-74991285-74995997-74996293__0:0:0'
 'ENSG00000104823.9;ECH1;chr19-38817315-38817364-38817064-38817129-38831077-38831166__0:0:0']
Test barcodes: ['ENSG00000133106.15;EPSTI1;chr13-42963254-42963338-42953947-42954021-42969093-42969177__0:0:0'
 'ENSG00000183671.12;GPR1;chr2-206203893-206203948-206176565-206177275-206213306-206213371__0:0:0'
 'ENSG00000145741.16;BTF3;chr5-73504346-73504403-73502487-73503117-73505191-73505435__0:0:0'
 'ENSG00000124596.17;OARD1;chr6-41075266-41075349-41071595-41071675-41097712-41097787__0:0:0'
 'ENSG00000096063.16;SRPK1;chr6-35920467-35920528-35890894-35891013-35921043-35921072__0:0:0']
Train

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Train barcodes: ['ENSG00000138152.10;BTBD16;chr10-122297767-122297837-122291079-122291194-122307188-122307308__0:0:0'
 'ENSG00000123411.15;IKZF4;chr12-56018125-56018177-56007668-56007734-56025053-56025158__0:0:0'
 'ENSG00000146263.12;MMS22L;chr6-97246627-97246690-97233860-97233980-97267871-97268002__0:0:0'
 'ENSG00000122126.17;OCRL;chrX-129567253-129567363-129565771-129565883-129569263-129569399__0:0:0'
 'ENSG00000137845.15;ADAM10;chr15-58643885-58643978-58640776-58640960-58646054-58646204__0:0:0']
Test barcodes: ['ENSG00000066923.18;STAG3;chr7-100182722-100182839-100182089-100182192-100189444-100189596__0:0:0'
 'ENSG00000100003.18;SEC14L2;chr22-30406341-30406385-30399642-30399718-30407094-30407154__0:0:0'
 'ENSG00000100344.11;PNPLA3;chr22-43928823-43928889-43926934-43927167-43932877-43933087__0:0:0'
 'ENSG00000183431.12;SF3A3;chr1-37978987-37979055-37976883-37976953-37979464-37979533__0:0:0'
 'ENSG00000021826.17;CPS1;chr2-210668185-210668284-210663122-210663197-210674901-210674961__0:

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Train barcodes: ['ENSG00000223458.2;LMO7DN-IT1;chr13-75878714-75878794-75876885-75876965-75879703-75879782__0:0:0'
 'ENSG00000164692.18;COL1A2;chr7-94409722-94409821-94409563-94409608-94410241-94410295__0:0:0'
 'ENSG00000100033.17;PRODH;chr22-18921341-18921423-18919705-18919798-18925050-18925200__0:0:0'
 'ENSG00000172995.17;ARPP21;chr3-35708968-35709070-35706973-35707082-35715438-35715476__0:0:0'
 'ENSG00000145700.10;ANKRD31;chr5-75112512-75112600-75107520-75107617-75116565-75116681__0:0:0']
Test barcodes: ['ENSG00000164134.13;NAA15;chr4-139354025-139354098-139351504-139351611-139359742-139359857__0:0:0'
 'ENSG00000136878.14;USP20;chr9-129856306-129856360-129852539-129852636-129858049-129858112__0:0:0'
 'ENSG00000177830.18;CHID1;chr11-899339-899401-893426-893519-900003-900110__0:0:0'
 'ENSG00000262879.6;RP11-156P1.3;chr17-47007839-47007884-46998699-46998817-47021581-47021721__0:0:0'
 'ENSG00000149403.13;GRIK4;chr11-120898531-120898639-120875138-120875243-120905289-120905493__0:0:0']
Tr

...

Train barcodes: ['ENSG00000134690.11;CDCA8;chr1-37706977-37707064-37703251-37703347-37708321-37709713__0:0:0'
 'ENSG00000100578.18;KIAA0586;chr14-58512521-58512627-58508554-58508709-58540070-58540136__0:0:0'
 'ENSG00000102001.13;CACNA1F;chrX-49214158-49214269-49213818-49213902-49215085-49215244__0:0:0'
 'ENSG00000167306.20;MYO5B;chr18-49990438-49990520-49980443-49980553-49992287-49992431__0:0:0'
 'ENSG00000198677.12;TTC37;chr5-95506933-95506992-95503813-95503938-95513570-95513645__-19:0:0']
Test barcodes: ['ENSG00000141030.13;COPS3;chr17-17261599-17261692-17260373-17260474-17261965-17262106__0:0:0'
 'ENSG00000065413.20;ANKRD44;chr2-197187022-197187106-197147026-197147105-197310577-197310646__0:0:0'
 'ENSG00000001617.12;SEMA3F;chr3-50174051-50174114-50173792-50173953-50174230-50174350__0:0:0'
 'ENSG00000148187.18;MRRF;chr9-122291748-122291840-122270807-122271075-122322539-122323092__0:0:0'
 'ENSG00000131966.14;ACTR10;chr14-58207935-58208018-58202854-58202927-58208998-58209107__0:0:0']
T

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Train barcodes: ['ENSG00000068971.14;PPP2R5B;chr11-64928065-64928158-64927733-64927903-64928294-64928425__0:0:0'
 'ENSG00000123240.17;OPTN;chr10-13116266-13116340-13112452-13112635-13118887-13118980__0:-57:0'
 'ENSG00000272281.6;TPTE2P2;chr13-52229325-52229431-52227004-52227096-52253744-52253833__0:0:0'
 'ENSG00000047849.22;MAP4;chr3-47867245-47867338-47857430-47857512-47869213-47869327__0:0:0'
 'ENSG00000144867.13;SRPRB;chr3-133807745-133807823-133806552-133806703-133811116-133811199__0:0:0']
Test barcodes: ['ENSG00000105879.12;CBLL1;chr7-107753894-107753978-107753410-107753511-107758142-107758571__0:0:0'
 'ENSG00000140943.17;MBTPS1;chr16-84091731-84091848-84090874-84090942-84093710-84093821__0:0:0'
 'ENSG00000129460.16;NGDN;chr14-23470905-23470977-23469743-23470101-23475170-23475308__0:0:0'
 'ENSG00000205426.10;KRT81;chr12-52288360-52288456-52287983-52288148-52289214-52289275__0:0:0'
 'ENSG00000136811.16;ODF2;chr9-128492425-128492536-128487889-128488025-128492700-128492805__0:0:0']
T

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


Average across all seeds:
Pearson Correlation: 0.7270 ± 0.0044
Spearman Correlation: 0.7328 ± 0.0056


# For RBP_only

In [38]:
# import os
# output_dir = "/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/"
# seed_set = range(10)
# all_correlations = []

# for seed in seed_set:
#     random.seed(seed)
#     np.random.seed(seed)    
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)

#     # Create train/test split using current seed
#     train_dataset, test_dataset = split_dataset_by_barcode(dataset, test_size=0.2, random_state=seed)
#     print(f"Seed {seed} - Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8)
#     test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=8)

#     model_barcode = Predictor_barcode()
#     model_barcode.load_state_dict(torch.load(f"/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/model_barcode_PSI_no_mut_5epoch_seed{seed}.pth"))
#     model_barcode.eval()
#     model_barcode = model_barcode.to(device)

#     MSELoss = nn.MSELoss()
#     optimizer = torch.optim.Adam(model_barcode.parameters(), lr=lr)

#     model_gene_barcode = Predictor_gene_barcode()
#     model_gene_barcode.load_state_dict(torch.load(f"/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/model_barcode_gene_PSI_RBP_only_5epoch_seed{seed}.pth"))
#     model_gene_barcode.eval()
#     model_gene_barcode = model_gene_barcode.to(device)
    
#     predict_psi = []
#     real_psi = []
#     psi_residuals = []
#     celltypes = []
#     barcode_names = []
    
#     with torch.no_grad():
#         psi_loss = 0
#         recon_loss = 0
#         for i, (celltype, barcode_name, barcode, gx, psi) in enumerate(test_loader):
#             barcode = barcode.to(device).float()
#             barcode = barcode.permute(0, 2, 1)
#             gx = gx.to(device).float()
#             psi = psi.to(device).float()
        
#             condition = model_barcode.Barcode_model(barcode)
#             psi_pretrain = model_barcode(barcode)
        
#             psi_residual = model_gene_barcode(barcode, gx, condition)
#             psi_loss += MSELoss(psi_residual.squeeze(), psi-psi_pretrain.squeeze()).item()
            
#             predict_psi.extend(psi_residual.squeeze().cpu().numpy()+psi_pretrain.squeeze().cpu().numpy())
#             psi_residuals.extend(psi_residual.squeeze().cpu().numpy())
#             real_psi.extend(psi.cpu().numpy())
#             celltypes.extend(celltype)
#             barcode_names.extend(barcode_name)

#     pearson_correlation, _ = pearsonr(real_psi, predict_psi)
#     spearman_correlation, _ = spearmanr(real_psi, predict_psi)
#     all_correlations.append((pearson_correlation, spearman_correlation))
    
#     print(f"\nSeed {seed}:")
#     print(f"Pearson Correlation: {pearson_correlation:.4f}")
#     print(f"Spearman Correlation: {spearman_correlation:.4f}")

#     # Save predictions for this seed
#     psi_results_df = pd.DataFrame({
#         'Real_PSI': real_psi,
#         'Predicted_PSI': predict_psi,
#         'PSI_Residuals': psi_residuals,
#         'Celltype': celltypes,
#         'Barcode': barcode_names
#     })
#     psi_results_df.to_csv(os.path.join(output_dir, f'psi_predictions_barcode_gene_RBP_only_seed{seed}.csv'), index=False)
#     print(f"Saved PSI predictions to {os.path.join(output_dir, f'psi_predictions_barcode_gene_RBP_only_seed{seed}.csv')}")

#     # Create and save scatter plot for this seed
#     plt.figure(figsize=(8, 8), dpi=300)
#     plt.scatter(real_psi, predict_psi, alpha=0.5, s=5, c='#336695', edgecolors='none', marker='o', rasterized=True)
#     plt.plot([-10, 10], [-10, 10], 'k--', alpha=0.5, zorder=0)
#     plt.xlabel("Observed PSI", fontsize=14, fontweight='bold')
#     plt.ylabel("Predicted PSI", fontsize=14, fontweight='bold')
#     plt.xticks(fontsize=12)
#     plt.yticks(fontsize=12)
#     plt.xlim(min(real_psi), max(real_psi))
#     plt.ylim(min(predict_psi), max(predict_psi))
#     plt.gca().spines['right'].set_visible(False)
#     plt.gca().spines['top'].set_visible(False)
#     plt.gca().spines['left'].set_linewidth(1.5)
#     plt.gca().spines['bottom'].set_linewidth(1.5)
#     plt.title(f"Prediction Performance on Wild-type Data (Seed {seed})", fontsize=16, fontweight='bold', pad=20)
#     stats_text = f"Pearson r = {pearson_correlation:.3f}\nSpearman ρ = {spearman_correlation:.3f}"
#     plt.text(0.05, 0.95, stats_text,
#              transform=plt.gca().transAxes,
#              bbox=dict(facecolor='white', edgecolor='none', alpha=0.7),
#              fontsize=12,
#              verticalalignment='top')
#     plt.tight_layout()
#     plt.savefig(os.path.join(output_dir, f'predicted_vs_real_psi_gene_barcode_RBP_only_seed{seed}.pdf'), 
#                 bbox_inches='tight', 
#                 dpi=300)
#     plt.close()

# # After all seeds are processed, create summary statistics and plots
# pearson_values = [c[0] for c in all_correlations]
# spearman_values = [c[1] for c in all_correlations]

# # Save correlation values to CSV
# correlation_df = pd.DataFrame({
#     'Pearson': pearson_values,
#     'Spearman': spearman_values
# })
# correlation_df.to_csv(os.path.join(output_dir, 'correlation_values_model_barcode_gene_RBP_only.csv'), index=False)

# # Calculate means and stds
# pearson_mean = np.mean(pearson_values)
# pearson_std = np.std(pearson_values)
# spearman_mean = np.mean(spearman_values)
# spearman_std = np.std(spearman_values)

# # Create and save box plot
# plt.figure(figsize=(8, 6))
# box_data = [pearson_values, spearman_values]
# plt.boxplot(box_data, labels=['Pearson', 'Spearman'])
# plt.ylabel('Correlation')
# plt.title('Correlation Scores Across Seeds')
# x_positions = [1, 2]
# for i, data in enumerate([pearson_values, spearman_values]):
#     plt.scatter([x_positions[i]] * len(data), data, alpha=0.5, color='black')
# plt.gca().spines['right'].set_visible(False)
# plt.gca().spines['top'].set_visible(False)
# print(f"\nAverage across all seeds:")
# print(f"Pearson Correlation: {pearson_mean:.4f} ± {pearson_std:.4f}")
# print(f"Spearman Correlation: {spearman_mean:.4f} ± {spearman_std:.4f}")
# plt.savefig(os.path.join(output_dir, 'correlation_boxplot.pdf'), 
#             bbox_inches='tight', 
#             dpi=300)
# plt.close()


# Evaluation for model_barcode

In [39]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
# Loop through all 10 seeds
all_correlations_mutagenesis = []
output_dir = "/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/"
seed_set = range(10)
for seed in seed_set:
    random.seed(seed)
    np.random.seed(seed)    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Create train/test split using current seed
    train_dataset, test_dataset = split_dataset_by_barcode(dataset, test_size=0.2, random_state=seed)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=8)
    
    model_barcode = Predictor_barcode()
    model_barcode.load_state_dict(torch.load(f"/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/model_barcode_PSI_no_mut_5epoch_seed{seed}.pth"))
    model_barcode.eval()
    model_barcode = model_barcode.to(device)

    predict_psi = []
    real_psi = []
    psi_residuals = []
    celltypes = []
    barcode_names = []
    with torch.no_grad():
        psi_loss = 0
        recon_loss = 0
        for i, (celltype, barcode_name, barcode, gx, psi) in enumerate(test_loader):
            barcode = barcode.to(device).float()
            barcode = barcode.permute(0, 2, 1)
            gx = gx.to(device).float()
            psi = psi.to(device).float()
        
            psi_pretrain = model_barcode(barcode)
            psi_loss += MSELoss(psi_pretrain.squeeze(), psi).item()
            
            predict_psi.extend(psi_pretrain.squeeze().cpu().numpy())
            psi_residuals.extend(psi_pretrain.squeeze().cpu().numpy())
            real_psi.extend(psi.cpu().numpy())
            celltypes.extend(celltype)
            barcode_names.extend(barcode_name)

    pearson_correlation, _ = pearsonr(real_psi, predict_psi)
    spearman_correlation, _ = spearmanr(real_psi, predict_psi)
    all_correlations_mutagenesis.append((pearson_correlation, spearman_correlation))
    
    print(f"Seed {seed}:")
    print(f"Pearson Correlation: {pearson_correlation:.4f}")
    print(f"Spearman Correlation: {spearman_correlation:.4f}")
    
    # Create scatter plot for each seed
    plt.figure(figsize=(8, 8), dpi=300)
    plt.scatter(real_psi, predict_psi, alpha=0.5, s=5, c='#336695', edgecolors='none', marker='o', rasterized=True)
    
    # Add diagonal line
    plt.plot([-10, 10], [-10, 10], 'k--', alpha=0.5, zorder=0)
    
    plt.xlabel("Observed PSI", fontsize=14, fontweight='bold')
    plt.ylabel("Predicted PSI", fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    
    # Set x and y axis limits to the range of the data
    plt.xlim(-10, 10)
    plt.ylim(min(predict_psi), max(predict_psi))
    
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['left'].set_linewidth(1.5)
    plt.gca().spines['bottom'].set_linewidth(1.5)
    
    plt.title(f"Prediction Performance on Wild-type Data (Seed {seed})", fontsize=16, fontweight='bold', pad=20)
    
    # Add correlation stats in a box
    stats_text = f"Pearson r = {pearson_correlation:.3f}\nSpearman ρ = {spearman_correlation:.3f}"
    plt.text(0.05, 0.95, stats_text,
             transform=plt.gca().transAxes,
             bbox=dict(facecolor='white', edgecolor='none', alpha=0.7),
             fontsize=12,
             verticalalignment='top')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'predicted_vs_real_psi_barcode_seed{seed}.pdf'), 
                bbox_inches='tight', 
                dpi=300)
    plt.close()

    # Create a DataFrame to store the predictions and actual values for this seed
    predictions_df = pd.DataFrame({
        'Real_PSI': real_psi,
        'Predicted_PSI': predict_psi,
        'Celltype': celltypes,
        'Barcode': barcode_names
    })

    # Save predictions to CSV for this seed
    predictions_csv_path = os.path.join(output_dir, f'predictions_barcode_seed{seed}.csv')
    predictions_df.to_csv(predictions_csv_path, index=False)
    print(f"Saved predictions to {predictions_csv_path}")

# Create box plot data
pearson_values = [c[0] for c in all_correlations_mutagenesis]
spearman_values = [c[1] for c in all_correlations_mutagenesis]

# Save correlation values to CSV
correlation_df = pd.DataFrame({
    'Pearson': pearson_values,
    'Spearman': spearman_values
})
correlation_df.to_csv(os.path.join(output_dir, 'correlation_values_model_barcode.csv'), index=False)

# Calculate means and stds
pearson_mean = np.mean(pearson_values)
pearson_std = np.std(pearson_values)
spearman_mean = np.mean(spearman_values)
spearman_std = np.std(spearman_values)

# Create box plot
plt.figure(figsize=(8, 6))
box_data = [pearson_values, spearman_values]
plt.boxplot(box_data, tick_labels=['Pearson', 'Spearman'])
plt.ylabel('Correlation')
plt.title('Correlation Scores Across Seeds')

# Add individual points
x_positions = [1, 2]
for i, data in enumerate([pearson_values, spearman_values]):
    plt.scatter([x_positions[i]] * len(data), data, alpha=0.5, color='black')

plt.gca().spines['right'].set_visible(False)
plt.gca().spines['top'].set_visible(False)

print(f"\nAverage across all seeds:")
print(f"Pearson Correlation: {pearson_mean:.4f} ± {pearson_std:.4f}")
print(f"Spearman Correlation: {spearman_mean:.4f} ± {spearman_std:.4f}")

plt.savefig(os.path.join(output_dir, 'correlation_boxplot.pdf'), 
            bbox_inches='tight', 
            dpi=300)
plt.close()


Train barcodes: ['ENSG00000184156.17;KCNQ3;chr8-132134325-132134388-132132179-132132264-132137884-132138016__0:0:0'
 'ENSG00000105613.10;MAST1;chr19-12858361-12858441-12852327-12852395-12858530-12858739__0:0:0'
 'ENSG00000171109.19;MFN1;chr3-179365117-179365225-179364296-179364405-179367438-179367498__0:0:0'
 'ENSG00000203321.2;CARNMT1-AS1;chr9-74991620-74991674-74991164-74991285-74995997-74996293__0:0:0'
 'ENSG00000104823.9;ECH1;chr19-38817315-38817364-38817064-38817129-38831077-38831166__0:0:0']
Test barcodes: ['ENSG00000133106.15;EPSTI1;chr13-42963254-42963338-42953947-42954021-42969093-42969177__0:0:0'
 'ENSG00000183671.12;GPR1;chr2-206203893-206203948-206176565-206177275-206213306-206213371__0:0:0'
 'ENSG00000145741.16;BTF3;chr5-73504346-73504403-73502487-73503117-73505191-73505435__0:0:0'
 'ENSG00000124596.17;OARD1;chr6-41075266-41075349-41071595-41071675-41097712-41097787__0:0:0'
 'ENSG00000096063.16;SRPK1;chr6-35920467-35920528-35890894-35891013-35921043-35921072__0:0:0']
Train

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Saved predictions to /home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/predictions_barcode_seed0.csv
Train barcodes: ['ENSG00000138152.10;BTBD16;chr10-122297767-122297837-122291079-122291194-122307188-122307308__0:0:0'
 'ENSG00000123411.15;IKZF4;chr12-56018125-56018177-56007668-56007734-56025053-56025158__0:0:0'
 'ENSG00000146263.12;MMS22L;chr6-97246627-97246690-97233860-97233980-97267871-97268002__0:0:0'
 'ENSG00000122126.17;OCRL;chrX-129567253-129567363-129565771-129565883-129569263-129569399__0:0:0'
 'ENSG00000137845.15;ADAM10;chr15-58643885-58643978-58640776-58640960-58646054-58646204__0:0:0']
Test barcodes: ['ENSG00000066923.18;STAG3;chr7-100182722-100182839-100182089-100182192-100189444-100189596__0:0:0'
 'ENSG00000100003.18;SEC14L2;chr22-30406341-30406385-30399642-30399718-30407094-30407154__0:0:0'
 'ENSG00000100344.11;PNPLA3;chr22-43928823-43928889-43926934-43927167-43932877-43933087__0:0:0'
 'ENSG00000183431.12;SF3A3;chr1-37978987-37979055-37976883-37976953-37979464

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Saved predictions to /home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/predictions_barcode_seed1.csv
Train barcodes: ['ENSG00000223458.2;LMO7DN-IT1;chr13-75878714-75878794-75876885-75876965-75879703-75879782__0:0:0'
 'ENSG00000164692.18;COL1A2;chr7-94409722-94409821-94409563-94409608-94410241-94410295__0:0:0'
 'ENSG00000100033.17;PRODH;chr22-18921341-18921423-18919705-18919798-18925050-18925200__0:0:0'
 'ENSG00000172995.17;ARPP21;chr3-35708968-35709070-35706973-35707082-35715438-35715476__0:0:0'
 'ENSG00000145700.10;ANKRD31;chr5-75112512-75112600-75107520-75107617-75116565-75116681__0:0:0']
Test barcodes: ['ENSG00000164134.13;NAA15;chr4-139354025-139354098-139351504-139351611-139359742-139359857__0:0:0'
 'ENSG00000136878.14;USP20;chr9-129856306-129856360-129852539-129852636-129858049-129858112__0:0:0'
 'ENSG00000177830.18;CHID1;chr11-899339-899401-893426-893519-900003-900110__0:0:0'
 'ENSG00000262879.6;RP11-156P1.3;chr17-47007839-47007884-46998699-46998817-47021581-47021721_

...

Saved predictions to /home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/predictions_barcode_seed7.csv
Train barcodes: ['ENSG00000134690.11;CDCA8;chr1-37706977-37707064-37703251-37703347-37708321-37709713__0:0:0'
 'ENSG00000100578.18;KIAA0586;chr14-58512521-58512627-58508554-58508709-58540070-58540136__0:0:0'
 'ENSG00000102001.13;CACNA1F;chrX-49214158-49214269-49213818-49213902-49215085-49215244__0:0:0'
 'ENSG00000167306.20;MYO5B;chr18-49990438-49990520-49980443-49980553-49992287-49992431__0:0:0'
 'ENSG00000198677.12;TTC37;chr5-95506933-95506992-95503813-95503938-95513570-95513645__-19:0:0']
Test barcodes: ['ENSG00000141030.13;COPS3;chr17-17261599-17261692-17260373-17260474-17261965-17262106__0:0:0'
 'ENSG00000065413.20;ANKRD44;chr2-197187022-197187106-197147026-197147105-197310577-197310646__0:0:0'
 'ENSG00000001617.12;SEMA3F;chr3-50174051-50174114-50173792-50173953-50174230-50174350__0:0:0'
 'ENSG00000148187.18;MRRF;chr9-122291748-122291840-122270807-122271075-122322539-1223

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Saved predictions to /home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/predictions_barcode_seed8.csv
Train barcodes: ['ENSG00000068971.14;PPP2R5B;chr11-64928065-64928158-64927733-64927903-64928294-64928425__0:0:0'
 'ENSG00000123240.17;OPTN;chr10-13116266-13116340-13112452-13112635-13118887-13118980__0:-57:0'
 'ENSG00000272281.6;TPTE2P2;chr13-52229325-52229431-52227004-52227096-52253744-52253833__0:0:0'
 'ENSG00000047849.22;MAP4;chr3-47867245-47867338-47857430-47857512-47869213-47869327__0:0:0'
 'ENSG00000144867.13;SRPRB;chr3-133807745-133807823-133806552-133806703-133811116-133811199__0:0:0']
Test barcodes: ['ENSG00000105879.12;CBLL1;chr7-107753894-107753978-107753410-107753511-107758142-107758571__0:0:0'
 'ENSG00000140943.17;MBTPS1;chr16-84091731-84091848-84090874-84090942-84093710-84093821__0:0:0'
 'ENSG00000129460.16;NGDN;chr14-23470905-23470977-23469743-23470101-23475170-23475308__0:0:0'
 'ENSG00000205426.10;KRT81;chr12-52288360-52288456-52287983-52288148-52289214-522892

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Saved predictions to /home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/predictions_barcode_seed9.csv

Average across all seeds:
Pearson Correlation: 0.6971 ± 0.0049
Spearman Correlation: 0.7011 ± 0.0054


In [40]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


# Set publication-quality style
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 12,
    'axes.linewidth': 1.5,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'axes.titlesize': 16,
    'pdf.fonttype': 42
})
sns.set_style("ticks")

# Read correlation data
correlation_barcode_path = "/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/correlation_values_model_barcode.csv"
correlation_gene_barcode_path = "/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/correlation_values_model_barcode_gene.csv"
correlation_gene_barcode_RBP_only_path = "/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/correlation_values_model_barcode_gene_RBP_only.csv"
# Load the data
barcode_df = pd.read_csv(correlation_barcode_path)
gene_barcode_df = pd.read_csv(correlation_gene_barcode_path)
gene_barcode_RBP_only_df = pd.read_csv(correlation_gene_barcode_RBP_only_path)

# Create figure for comparison
fig, ax = plt.subplots(figsize=(6, 4), dpi=300)

# Create box plot data - Pearson only
box_data = [
    barcode_df['Pearson'],
    gene_barcode_RBP_only_df['Pearson'],
    gene_barcode_df['Pearson']
]

# Create box plot with custom colors
boxprops = dict(linewidth=2)
whiskerprops = dict(linewidth=2)
capprops = dict(linewidth=2)
medianprops = dict(linewidth=2, color='#444444')

bp = ax.boxplot(box_data, 
                patch_artist=True,
                boxprops=boxprops,
                whiskerprops=whiskerprops,
                capprops=capprops,
                medianprops=medianprops)

# Customize box colors
colors = ['#3498db', '#e74c3c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add individual points with jitter
for i, data in enumerate(box_data):
    # Add jitter to x position - reduced jitter range and centered around the box position
    x = np.random.normal(0, 0.03, size=len(data)) + (i+1)
    ax.scatter(x, data, alpha=0.7, s=50, color='black', edgecolor='white', linewidth=0.5)

# Calculate and display means and stds
for i, (name, df) in enumerate([
    ('Barcode Pearson', barcode_df),
    ('Gene Barcode Pearson', gene_barcode_df),
    ('Gene Barcode RBP_only Pearson', gene_barcode_RBP_only_df)
]):
    mean = np.mean(df['Pearson'])
    std = np.std(df['Pearson'])
    print(f"{name}: {mean:.4f} ± {std:.4f}")

# Customize plot appearance
ax.set_ylabel('Pearson Correlation', fontweight='bold')
ax.set_title('Pearson Correlation Comparison', fontweight='bold')
ax.set_xticklabels(['Sequence Only', 'With RBP Expression', 'With All Gene Expression'], 
                   fontweight='bold')

# Set y-axis range from 0.5 to 0.8
ax.set_ylim(bottom=0.6, top=0.8)

# Remove top and right spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

# Add grid for y-axis only
ax.yaxis.grid(True, linestyle='--', alpha=0.7)

# Adjust layout and save
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'correlation_boxplot_comparison.pdf'), bbox_inches='tight')
plt.close()

Barcode Pearson: 0.6971 ± 0.0049
Gene Barcode Pearson: 0.7302 ± 0.0063
Gene Barcode RBP_only Pearson: 0.7296 ± 0.0044


# Correlation boxplot for the finetune only with the variable sequences

In [41]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
# Loop through all 10 seeds
all_correlations_mutagenesis = []
output_dir = "/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/"
seed_set = range(10)
for seed in seed_set:
    random.seed(seed)
    np.random.seed(seed)    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Create train/test split using current seed
    train_dataset, test_dataset = split_dataset_by_barcode(dataset, test_size=0.2, random_state=seed)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=8)
    
    model_barcode = Predictor_barcode()
    model_barcode.load_state_dict(torch.load(f"/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/model_barcode_PSI_no_mut_5epoch_seed{seed}.pth"))
    model_barcode.eval()
    model_barcode = model_barcode.to(device)

    predict_psi = []
    real_psi = []
    psi_residuals = []
    celltypes = []
    barcode_names = []
    with torch.no_grad():
        psi_loss = 0
        recon_loss = 0
        for i, (celltype, barcode_name, barcode, gx, psi) in enumerate(test_loader):
            barcode = barcode.to(device).float()
            barcode = barcode.permute(0, 2, 1)
            gx = gx.to(device).float()
            psi = psi.to(device).float()
        
            psi_pretrain = model_barcode(barcode)
            psi_loss += MSELoss(psi_pretrain.squeeze(), psi).item()
            
            predict_psi.extend(psi_pretrain.squeeze().cpu().numpy())
            psi_residuals.extend(psi_pretrain.squeeze().cpu().numpy())
            real_psi.extend(psi.cpu().numpy())
            celltypes.extend(celltype)
            barcode_names.extend(barcode_name)

    pearson_correlation, _ = pearsonr(real_psi, predict_psi)
    spearman_correlation, _ = spearmanr(real_psi, predict_psi)
    all_correlations_mutagenesis.append((pearson_correlation, spearman_correlation))
    
    print(f"Seed {seed}:")
    print(f"Pearson Correlation: {pearson_correlation:.4f}")
    print(f"Spearman Correlation: {spearman_correlation:.4f}")
    
    # Create scatter plot for each seed
    plt.figure(figsize=(8, 8), dpi=300)
    plt.scatter(real_psi, predict_psi, alpha=0.5, s=5, c='#336695', edgecolors='none', marker='o', rasterized=True)
    
    # Add diagonal line
    plt.plot([-10, 10], [-10, 10], 'k--', alpha=0.5, zorder=0)
    
    plt.xlabel("Observed PSI", fontsize=14, fontweight='bold')
    plt.ylabel("Predicted PSI", fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    
    # Set x and y axis limits to the range of the data
    plt.xlim(min(real_psi), max(real_psi))
    plt.ylim(min(predict_psi), max(predict_psi))
    
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['left'].set_linewidth(1.5)
    plt.gca().spines['bottom'].set_linewidth(1.5)
    
    plt.title(f"Prediction Performance on Wild-type Data (Seed {seed})", fontsize=16, fontweight='bold', pad=20)
    
    # Add correlation stats in a box
    stats_text = f"Pearson r = {pearson_correlation:.3f}\nSpearman ρ = {spearman_correlation:.3f}"
    plt.text(0.05, 0.95, stats_text,
             transform=plt.gca().transAxes,
             bbox=dict(facecolor='white', edgecolor='none', alpha=0.7),
             fontsize=12,
             verticalalignment='top')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'predicted_vs_real_psi_barcode_with_high_var_only_seed{seed}.pdf'), 
                bbox_inches='tight', 
                dpi=300)
    plt.close()

    # Create a DataFrame to store the predictions and actual values for this seed
    predictions_df = pd.DataFrame({
        'Real_PSI': real_psi,
        'Predicted_PSI': predict_psi,
        'Celltype': celltypes,
        'Barcode': barcode_names
    })

    # Save predictions to CSV for this seed
    predictions_csv_path = os.path.join(output_dir, f'predictions_barcode_with_high_var_only_seed{seed}.csv')
    predictions_df.to_csv(predictions_csv_path, index=False)
    print(f"Saved predictions to {predictions_csv_path}")

# Create box plot data
pearson_values = [c[0] for c in all_correlations_mutagenesis]
spearman_values = [c[1] for c in all_correlations_mutagenesis]

# Save correlation values to CSV
correlation_df = pd.DataFrame({
    'Pearson': pearson_values,
    'Spearman': spearman_values
})
correlation_df.to_csv(os.path.join(output_dir, 'correlation_values_model_barcode_with_high_var_only.csv'), index=False)

# Calculate means and stds
pearson_mean = np.mean(pearson_values)
pearson_std = np.std(pearson_values)
spearman_mean = np.mean(spearman_values)
spearman_std = np.std(spearman_values)

# Create box plot
plt.figure(figsize=(8, 6))
box_data = [pearson_values, spearman_values]
plt.boxplot(box_data, tick_labels=['Pearson', 'Spearman'])
plt.ylabel('Correlation')
plt.title('Correlation Scores Across Seeds')

# Add individual points
x_positions = [1, 2]
for i, data in enumerate([pearson_values, spearman_values]):
    plt.scatter([x_positions[i]] * len(data), data, alpha=0.5, color='black')

plt.gca().spines['right'].set_visible(False)
plt.gca().spines['top'].set_visible(False)

print(f"\nAverage across all seeds:")
print(f"Pearson Correlation: {pearson_mean:.4f} ± {pearson_std:.4f}")
print(f"Spearman Correlation: {spearman_mean:.4f} ± {spearman_std:.4f}")

plt.savefig(os.path.join(output_dir, 'correlation_boxplot_with_high_var_only.pdf'), 
            bbox_inches='tight', 
            dpi=300)
plt.close()


Train barcodes: ['ENSG00000184156.17;KCNQ3;chr8-132134325-132134388-132132179-132132264-132137884-132138016__0:0:0'
 'ENSG00000105613.10;MAST1;chr19-12858361-12858441-12852327-12852395-12858530-12858739__0:0:0'
 'ENSG00000171109.19;MFN1;chr3-179365117-179365225-179364296-179364405-179367438-179367498__0:0:0'
 'ENSG00000203321.2;CARNMT1-AS1;chr9-74991620-74991674-74991164-74991285-74995997-74996293__0:0:0'
 'ENSG00000104823.9;ECH1;chr19-38817315-38817364-38817064-38817129-38831077-38831166__0:0:0']
Test barcodes: ['ENSG00000133106.15;EPSTI1;chr13-42963254-42963338-42953947-42954021-42969093-42969177__0:0:0'
 'ENSG00000183671.12;GPR1;chr2-206203893-206203948-206176565-206177275-206213306-206213371__0:0:0'
 'ENSG00000145741.16;BTF3;chr5-73504346-73504403-73502487-73503117-73505191-73505435__0:0:0'
 'ENSG00000124596.17;OARD1;chr6-41075266-41075349-41071595-41071675-41097712-41097787__0:0:0'
 'ENSG00000096063.16;SRPK1;chr6-35920467-35920528-35890894-35891013-35921043-35921072__0:0:0']
Train

# Now try to plot a heatmap of a few sequences across cell types.

In [42]:
from matplotlib.colors import LinearSegmentedColormap

# Read in the seed0 predictions
seed0_barcode_predictions = pd.read_csv('/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/predictions_barcode_seed0.csv')
seed0_gene_barcode_predictions = pd.read_csv('/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/psi_predictions_barcode_gene_seed0.csv')
seed0_gene_barcode_RBP_only_predictions = pd.read_csv('/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/psi_predictions_barcode_gene_RBP_only_seed0.csv')

# Convert log2(I/E) ratio back to PSI for each dataframe
for df in [seed0_barcode_predictions, seed0_gene_barcode_predictions, seed0_gene_barcode_RBP_only_predictions]:
    # Calculate I and E from log2(I/E)
    I = 1000 * (2**df['Real_PSI']) / (1 + 2**df['Real_PSI'])
    E = 1000 - I
    df['Real_PSI_converted'] = I / (I + E)
    
    # Do the same for predicted values
    I_pred = 1000 * (2**df['Predicted_PSI']) / (1 + 2**df['Predicted_PSI'])
    E_pred = 1000 - I_pred
    df['Predicted_PSI_converted'] = I_pred / (I_pred + E_pred)

# Get barcodes with at least 30 celltypes
barcodes_with_enough_celltypes = seed0_gene_barcode_predictions.groupby('Barcode').size()
barcodes_with_enough_celltypes = barcodes_with_enough_celltypes[barcodes_with_enough_celltypes >= 30].index

# Create custom colormap
colors = ["#4575B4", '#85B6D6', '#E2EFF2', '#FFE3B0', '#EF9651', '#D83629']
n_bins = 100
custom_cmap = LinearSegmentedColormap.from_list('custom_diverging', colors, N=n_bins)

def plot_heatmap(barcode_variance, title_suffix, filename_suffix):
    # Create a figure with subplots for each barcode
    fig, axes = plt.subplots(len(barcode_variance), 1, figsize=(15, 3*len(barcode_variance)))
    if len(barcode_variance) == 1:
        axes = [axes]

    for idx, (barcode, variance) in enumerate(barcode_variance.items()):
        # Get data for this barcode
        barcode_data = seed0_gene_barcode_predictions[seed0_gene_barcode_predictions['Barcode'] == barcode]
        
        # Create a 2xN matrix where N is number of cell types
        heatmap_data = np.vstack([
            barcode_data['Real_PSI'].values,
            barcode_data['Predicted_PSI'].values
        ])
        
        # Plot heatmap
        sns.heatmap(heatmap_data, 
                    ax=axes[idx],
                    cmap=custom_cmap,
                    vmin=0, 
                    vmax=1,
                    cbar_kws={'label': 'PSI'},
                    xticklabels=barcode_data['Celltype'].values,
                    yticklabels=['Real', 'Predicted'])
        
        # Add title with variance
        axes[idx].set_title(f'Barcode: {barcode}\nVariance: {variance:.4f}')
        
        # Rotate x-axis labels for better readability
        plt.setp(axes[idx].get_xticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'barcodes_heatmap_gene_barcode_{filename_suffix}.pdf'), bbox_inches='tight')
    plt.close()

# Plot high variance barcodes
barcode_high_variance = seed0_gene_barcode_predictions[seed0_gene_barcode_predictions['Barcode'].isin(barcodes_with_enough_celltypes)].groupby(['Barcode'])['Real_PSI_converted'].var().sort_values(ascending=False).head(20)
plot_heatmap(barcode_high_variance, 'High Variance', 'high_variance')

# Plot random sample barcodes
barcode_random = seed0_gene_barcode_predictions[seed0_gene_barcode_predictions['Barcode'].isin(barcodes_with_enough_celltypes)].groupby(['Barcode'])['Real_PSI_converted'].var().sample(n=20, random_state=42)
plot_heatmap(barcode_random, 'Random Sample', 'random_sample')
